In [2]:
# System Path #
import os
import sys 

# Add dsci_550_a1 to base path. Lets you project functions #
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Pandas #
import pandas as pd
import json
import re

# Runtime #
import time
from tqdm import tqdm 

# Iterators #
import collections
import ast
import random
from typing import Pattern
from itertools import chain
from collections import Counter

from dsci_550_a1.parsingFunctions import *



In [10]:
df = pd.read_csv("../data/processed/haunted_places_features_added.tab", sep = "\t")
df.head()

,city,country,description,location,state,state_abbrev,longitude,latitude,city_longitude,city_latitude,Audio_Evidence,Visual_Evidence,Haunted_Places_Date,Aerodrome_Intersections,Aerodrome_Count,Aerodrome_Proximity,Flight_Intersections,Flight_Intersection_Count,Flight_HighTraffic,Haunted_Places_Witness_Count
0,Ada,United States,ada witch sometimes see misty blue figure floa...,Ada Cemetery,Michigan,MI,-85.504893,42.962106,-85.495480,42.960727,True,True,"[datetime.datetime(2025, 1, 1, 0, 0), datetime...","[{'Airport_ID': 2878, 'Name': 'Somerville Airp...",3,True,"[{'Route_ID': 1011, 'Source_Airport': 'DFW', '...",56,True,7
1,Addison,United States,little girl killed suddenly waiting school bus...,North Adams Rd.,Michigan,MI,-84.381843,41.971425,-84.347168,41.986434,False,True,"[datetime.datetime(2025, 1, 1, 0, 0)]",[],0,False,"[{'Route_ID': 5228, 'Source_Airport': 'MKE', '...",3,False,1
2,Adrian,United States,take gorman rd . west towards sand creek come ...,Ghost Trestle,Michigan,MI,-84.035656,41.904538,-84.037166,41.897547,False,False,"[datetime.datetime(2025, 1, 1, 0, 0)]",[],0,False,"[{'Route_ID': 3764, 'Source_Airport': 'DTW', '...",5,False,0
3,Adrian,United States,1970 's one room room 211 old section dorms ex...,Siena Heights University,Michigan,MI,-84.017565,41.905712,-84.037166,41.897547,False,True,"[datetime.datetime(2025, 5, 1, 0, 0)]",[],0,False,"[{'Route_ID': 3764, 'Source_Airport': 'DTW', '...",5,False,2
4,Albion,United States,kappa delta sorority kappa delta sorority haun...,Albion College,Michigan,MI,-84.745177,42.244006,-84.753030,42.243097,True,False,"[datetime.datetime(2025, 1, 1, 0, 0)]",[],0,False,"[{'Route_ID': 924, 'Source_Airport': 'DCA', 'D...",22,True,1


In [7]:
# # Define groups of keywords with regex patterns
# event_groups = {
#     "Violence": [
#         r"murder(?:ed|s)?",         # murder, murdered, murders
#         r"kill(?:ed|s|ing)?",        # kill, killed, kills, killing
#         r"homicide",                # homicide
#         r"death(?:s)?",             # death, deaths
#         r"died",                    # died
#         r"slain",                   # slain
#         r"massacre(?:d|s)?",         # massacre, massacred, massacres
#         r"execute(?:d|s|ing)?",      # execute, executed, executes, executing
#         r"crime(?:s)?",             # crime, crimes
#         r"strangl(?:e|ed|es|ing)",   # strangle, strangled, strangles, strangling
#         r"stab(?:bed|s|bing)?",      # stab, stabbed, stabs, stabbing
#         r"shot",                    # shot
#         r"suicide(?:s)?"            # suicide, suicides
#     ],
#     "Supernatural": [
#         r"supernatural",            # supernatural
#         r"paranormal",              # paranormal
#         r"haunt(?:ed|ing|s)?",       # haunt, haunted, haunting, haunts
#         r"curse(?:d|s|ing)?",        # curse, cursed, curses, cursing
#         r"evil",                    # evil
#         r"possess(?:ion|ed|es|ing)?",# possession, possessed, possesses, possessing
#         r"ghost(?:ly|s)?",          # ghost, ghostly, ghosts
#         r"poltergeist",             # poltergeist
#         r"eerie",                   # eerie
#         r"demon(?:ic|strated|s)?",   # demon, demonic, demons
#         r"mysterious",              # mysterious
#         r"occult",                  # occult
#         r"unexplained",             # unexplained
#         r"witchcraft",              # witchcraft
#         r"witch(?:es)?"            # witch, witches
#     ],
#     "Accident/Disaster": [
#         r"accident(?:al|s)?",       # accident, accidental, accidents
#         r"traged(?:y|ies)",         # tragedy, tragedies
#         r"drown(?:ing|ed|s)?",       # drown, drowning, drowns, drowned
#         r"fire(?:d|s|ing)?",         # fire, fired, fires, firing
#         r"explos(?:ion|ions|ive|ed)",# explosion, explosions, explosive, exploded
#         r"disaster(?:s)?",          # disaster, disasters
#         r"fall(?:en|s|ing)?",        # fall, fallen, falls, falling
#         r"collaps(?:ed|es|ing)?",    # collapse, collapsed, collapses, collapsing
#         r"burn(?:ed|s|ing)?",        # burn, burned, burns, burning
#         r"suffocat(?:ion|e|es|ing)?",# suffocation, suffocate, suffocates, suffocating, suffocated
#         r"lost",                    # lost
#         r"catastroph(?:e|es)",       # catastrophe, catastrophes
#         r"avalanche(?:s)?",         # avalanche, avalanches
#         r"shipwreck(?:ed|s)?",       # shipwreck, shipwrecked, shipwrecks
#         r"landslide(?:s)?"          # landslide, landslides
#     ]
# }
patterns = json.load(open("../data/keywords/airborne_keywords.json"))
# patterns["Precompiled_Regex"]

In [ ]:
# Output Df
outfile = "../data/processed/haunted_places_features_added.tab"

# Reading CSV
df = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep = "\t")

# Feature Names
feature_names = ["Event_Type"]

patterns = json.load(open("../data/keywords/airbone_keywords.json"))
check_regex()

def classify_event(text):
    if not isinstance(text, str):
        return "Unknown"

    text = text.lower()
    triggered_groups = set()

    # Check every keyword in every group
    for group, patterns in event_groups.items():
        for pattern in patterns:
            if re.search(rf"\b{pattern}\b", text):
                triggered_groups.add(group)


    if not triggered_groups:
        return "Unknown"

    # Return all matched groups
    return ", ".join(sorted(triggered_groups))

# Start timing
start = time.time()

# Apply classification to the "description" column.
df["Event Type"] = df["description"].apply(classify_event)

# Stop timing
end = time.time()

# Count occurrences
counts = df["Event Type"].value_counts()
unknown_counts = counts.get("Unknown", 0)

extract_printout = [(category, count) for category, count in counts.items()]
print("-" * 100, "Extraction Completed", "-" * 100)
print(f"Extraction Took: {end - start:.6f} seconds\n")
print("\n".join([f"{category}: {count}" for category, count in extract_printout]))
print("-" * 100)